Trening HiFi-GAN w środowisku Google Colab

In [ ]:
! git clone https://github.com/NVIDIA/apex
! pip install -v --disable-pip-version-check --no-cache-dir --global-option="--cpp_ext" --global-option="--cuda_ext" --no-build-isolation ./apex

In [ ]:
! git clone https://github.com/NVIDIA/DeepLearningExamples.git
%cd DeepLearningExamples/PyTorch/SpeechSynthesis/HiFiGAN

In [ ]:
! pip install -r requirements.txt

In [4]:
! mkdir results data/dataset checkpoints

In [ ]:
! gdown 1gvbrV3VYSzOMZhphZYZDMGqtM8p59v6h -O /content/train_data.zip    # dane treningowe
! gdown 1V86B_b6ffXlFN10P0Q57SAv8CEEHuSTm -O /content/train_data_mels.zip   # dane treningowe - mel spektrogramy (do fine-tuningu)
! gdown 1jmODVz9gizWv74zRp7gTSA8UW_14T9-W -O ./checkpoints/hifigan_gen_checkpoint_6500.pt   # checkpoint generatora z treningu na LJ Speech
! gdown 1dhOImFLBKx1v6AhHBj-K8PKegz5Mg_mr -O ./checkpoints/hifigan_discrim_checkpoint_6500.pt   # checkpoint dyskryminatora z treningu na LJ Speech

In [ ]:
! unzip -q /content/train_data.zip -d ./data/dataset
! unzip -q /content/train_data_mels.zip -d ./data/dataset/train_data/
! mv ./data/dataset/train_data/metadata_train.txt ./data/filelists/metadata_train.txt
! mv ./data/dataset/train_data/metadata_val.txt ./data/filelists/metadata_val.txt
! rm ./data/dataset/train_data/metadata.csv

In [ ]:
# # trening od zera
# ! python -W "ignore::FutureWarning" train.py \
# --cuda \
# --dataset_path ./data/dataset/train_data/ \
# --training_files ./data/filelists/metadata_train.txt \
# --validation_files ./data/filelists/metadata_val.txt \
# --output ./results \
# --checkpoint_interval 5 \
# --epochs 50 \
# --batch_size 16 \
# --learning_rate 0.0003 \
# --lr_decay 0.9998 \
# --validation_interval 1 \
# --step_logs_interval 20 \
# --resblock_kernel_sizes [3,7,11]

In [ ]:
! cp checkpoints/hifigan_*_checkpoint_6500.pt results/  # checkpointy do fine-tuningu

In [ ]:
# fine-tuning
! python -W "ignore::FutureWarning" train.py \
--cuda \
--dataset_path ./data/dataset/train_data/ \
--training_files ./data/filelists/metadata_train.txt \
--validation_files ./data/filelists/metadata_val.txt \
--output ./results \
--checkpoint_interval 5 \
--epochs 6550 \
--batch_size 16 \
--learning_rate 3e-5 \
--lr_decay 0.9998 \
--validation_interval 1 \
--samples_interval 100 \
--step_logs_interval 20 \
--resume \
--resblock_kernel_sizes [3,7,11] \
--fine_tuning \
--fine_tune_lr_factor 0.1 \
--input_mels_dir ./data/dataset/train_data/mels \
--epochs_this_job 10

In [ ]:
! zip results.zip results/train/* results/val/* results/hifigan_gen* results/nvlog* results/hifigan_discrim_checkpoint_6550.pt